# 远端重建九列并逐日审计

本 Notebook 是唯一主运行文件。它会隔离运行此前两个最终上传包，读取远端现货和远端三状态基准，重建两个五列结果，合并为 9 列，并在输出栏直接展示三状态逐日一致性、九列逐列一致性和最终 PASS/FAIL。

默认路径已经直接填入此前最终上传包中的路径；如果远端挂载位置不同，再通过环境变量覆盖即可。

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

PACKAGE_ROOT = Path.cwd()
if not (PACKAGE_ROOT / 'remote_rebuild.py').is_file():
    PACKAGE_ROOT = next(parent for parent in [PACKAGE_ROOT, *PACKAGE_ROOT.parents] if (parent / 'remote_rebuild.py').is_file())
import sys
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))
from remote_rebuild import run_remote_audit

# 直接使用此前最终上传包中的远端路径；环境变量存在时允许覆盖。
DEFAULT_SPOT_PATH = '/home/hzy/cta/IC数据更新*最终固化版/现货最终版/CSI500_SPOT_md_eod_raw*最终版.parquet'
DEFAULT_THREE_STATE_PATH = '/home/hzy/cta/三状态冻结/IC_1545_three_state_and_downside_warning.csv'
SPOT_PATH = os.environ.get('COMPANY_SPOT_PATH') or DEFAULT_SPOT_PATH
THREE_STATE_PATH = os.environ.get('REMOTE_THREE_STATE_PATH') or DEFAULT_THREE_STATE_PATH
OUTPUT_DIR = os.environ.get('REMOTE_OUTPUT_DIR') or None

print('远端现货路径:', SPOT_PATH)
print('远端三状态路径:', THREE_STATE_PATH)
print('输出目录:', OUTPUT_DIR or str(PACKAGE_ROOT / '远端输出'))

results = run_remote_audit(spot_path=SPOT_PATH, three_state_path=THREE_STATE_PATH, output_dir=OUTPUT_DIR)
manifest = results['manifest']

## 1. 最终结论与输入审计

In [ ]:
state_audit = manifest['state_audit']
local_audit = manifest['local_nine_audit']
summary = pd.DataFrame([
    {'检查项': '1545/长0/远端三状态逐日一致', '结果': state_audit['all_three_state_checks_exact']},
    {'检查项': '远端1545五列 vs 本地审计附件逐列一致', '结果': manifest['local_five_audits']['1545']['no_date_or_value_difference']},
    {'检查项': '远端长0五列 vs 本地审计附件逐列一致', '结果': manifest['local_five_audits']['long0']['no_date_or_value_difference']},
    {'检查项': '远端九列 vs 本地参考九列逐列一致', '结果': local_audit['no_date_or_value_difference']},
    {'检查项': '最终可交付 PASS', '结果': manifest['success']},
    {'检查项': '九列行数', '结果': manifest['merge_audit']['rows']},
    {'检查项': '日期范围', '结果': f"{manifest['merge_audit']['date_min']} -> {manifest['merge_audit']['date_max']}"},
    {'检查项': '远端三状态共同日期行数', '结果': state_audit['common_rows_with_nine_grid']},
    {'检查项': '远端独有日期行数', '结果': state_audit['remote_only_rows']},
    {'检查项': '重建独有日期行数', '结果': state_audit['generated_only_rows']},
])
display(summary)

display(pd.DataFrame([manifest['merge_audit']]))
display(pd.DataFrame([state_audit]))
display(pd.DataFrame([local_audit]))
print('PASS' if manifest['success'] else 'FAIL：请先查看下方差异表和运行日志')

## 2. 三状态逐日对比

In [ ]:
state_compare = results['state_compare']
diff = state_compare.loc[state_compare['any_difference']].copy()
print('三状态差异行数:', len(diff))
if diff.empty:
    print('三状态逐日完全一致。')
else:
    display(diff.head(200))
print('状态计数：')
display(pd.DataFrame({
    '来源': ['1545重建', '长0重建', '远端基准'],
    '-1': [state_audit['generated_1545_counts'].get('-1', 0), state_audit['generated_long0_counts'].get('-1', 0), state_audit['remote_counts'].get('-1', 0)],
    '0': [state_audit['generated_1545_counts'].get('0', 0), state_audit['generated_long0_counts'].get('0', 0), state_audit['remote_counts'].get('0', 0)],
    '+1': [state_audit['generated_1545_counts'].get('1', 0), state_audit['generated_long0_counts'].get('1', 0), state_audit['remote_counts'].get('1', 0)],
}))

## 3. 九列结果预览与本地逐列对比

In [ ]:
display(results['nine'].head(10))
display(results['nine'].tail(10))
for label, frame in [('远端1545五列 vs 本地审计附件', results['local_1545_compare']), ('远端长0五列 vs 本地审计附件', results['local_long0_compare'])]:
    five_diff = frame.loc[frame['row_status'] != 'MATCH'].copy()
    print(label, '差异行数:', len(five_diff))
    if not five_diff.empty:
        display(five_diff.head(200))
local_compare = results['local_compare']
local_diff = local_compare.loc[local_compare['row_status'] != 'MATCH'].copy()
print('九列逐列差异行数:', len(local_diff))
if local_diff.empty:
    print('远端九列与本地参考九列在共同日期上逐列完全一致。')
else:
    display(local_diff.head(200))
print('九列字段:', list(results['nine'].columns))

## 4. 输出文件和日志

In [ ]:
display(pd.DataFrame([manifest['generated_files']]).T.rename(columns={0: '路径'}))
print('两个冻结引擎的完整进度已在前面输出；日志也保存在输出目录中。')
print('最终状态:', 'PASS' if manifest['success'] else 'FAIL')